# 01 — Hillstrom: EDA and dataset understanding (Day 1)

Understand the data **before** any causal machinery. What is one row? What is
missing, duplicated, skewed or mislabelled? Which columns are inputs and which
are outcomes?

- Dataset: MineThatData Hillstrom e-mail experiment — 64,000 customers, 3 arms, 14-day window.
- Structured spec: [`docs/datasets/hillstrom.md`](../../docs/datasets/hillstrom.md)
- Vocabulary (arm, treatment, ...): [`docs/concepts.md`](../../docs/concepts.md)

**Scope.** This notebook is descriptive only. Randomization checks, SMD and the
ATE live in `02_hillstrom_randomization_ate.ipynb`, because those introduce causal
concepts that deserve their own space.

## 0. Setup

In [ ]:
# Make the repo root importable so `packages.*` resolves from the notebook.
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from packages.datasets.hillstrom import OUTCOMES, load_hillstrom_raw

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 3.5)

raw = load_hillstrom_raw()
print("repo root:", ROOT)

In [ ]:
raw.head(3)

## 1. Shape and dtypes

One row = one customer. 12 columns: 8 pre-treatment attributes, 1 treatment
assignment (`segment`), 3 post-treatment outcomes.

In [ ]:
print("shape:", raw.shape)
print(f"memory: {raw.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print("\ncolumn groups")
for name, cols in {
    "pre-treatment": ["recency", "history_segment", "history", "mens", "womens",
                      "zip_code", "newbie", "channel"],
    "treatment": ["segment"],
    "outcomes": OUTCOMES,
}.items():
    print(f"  {name:14s} {cols}")
raw.dtypes.to_frame("dtype")

**Finding.** `zip_code`, `channel`, `segment` and `history_segment` load as
`str`; the rest are numeric. `visit`/`conversion` are `int64` and should be read
as 0/1 flags, not counts.

## 2. Missingness

In [ ]:
missing = raw.isna().sum()
print("total missing cells:", int(missing.sum()))
print("blank strings:", int((raw.select_dtypes("str") == "").sum().sum()))
missing[missing > 0]

**Finding.** No missing values, no blank strings. Any `NaN` appearing later comes
from our own joins or casts, not from the source.

## 3. Duplicate rows

Worth checking early: identical rows could mean a broken export (double-counted
customers) or simply coarse features (many customers genuinely share the same
12 attributes).

In [ ]:
n_dup = int(raw.duplicated().sum())
print("rows that repeat an earlier row:", n_dup, f"({n_dup / len(raw):.1%})")
print("distinct rows:", raw.drop_duplicates().shape[0], "of", len(raw))

in_group = raw[raw.duplicated(keep=False)]
print("\nrows belonging to a duplicate group:", len(in_group))
print("distinct records behind them:", in_group.drop_duplicates().shape[0])
print("\nduplicate rows per arm (should be even if assignment is random):")
print(in_group["segment"].value_counts().to_string())

**Finding.** 6,562 rows (10.3%) exactly repeat another row; 57,438 rows are
distinct. The repeats are spread almost evenly across the three arms
(~2,500 each), which is what random assignment predicts and what a doubled export
would *not* look like — a bad join usually skews one arm.

**Verdict: keep them.** They are plausible distinct customers, and dropping them
would silently change the arm sizes and the outcome rates. Worth revisiting on
Day 2 if the model overfits identical rows.

## 4. Feature by feature

### 4.1 `recency` — months since last purchase

Integer 1–12 by construction.

In [ ]:
print(raw["recency"].describe().to_string())
counts = raw["recency"].value_counts().sort_index()
ax = counts.plot.bar(width=0.85)
ax.set_xlabel("recency (months)")
ax.set_ylabel("customers")
ax.set_title("recency distribution — note the spikes at 1 and 9-10")
plt.tight_layout()
plt.show()

**Finding.** Bounded 1–12, but clearly **not uniform**: a spike at 1 (8,952
customers) and a second bump at 9–10, with a dip around 8. Recency is a real
behavioural signal, not a flat ID-like column — useful for segments.

### 4.2 `history` — dollars spent in the past year

In [ ]:
print(raw["history"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).to_string())
print("skew:", round(raw["history"].skew(), 2))
ax = raw["history"].plot.hist(bins=60)
ax.set_xlabel("history ($)")
ax.set_title("history — heavily right-skewed")
plt.tight_layout()
plt.show()

**Finding.** Right-skewed (skew ≈ 2.4): median $158 but mean $242, with a long
tail to $3,346 and 1% of customers above ~$1,219. Any model using it as-is will be
dominated by a handful of big spenders — a candidate for a log transform on Day 2,
though tree models care less.

### 4.3 `history_segment` — the bucket label (and the comma trap)

This column and `history` look like two features but are one. Also: **buckets 6
and 7 contain embedded commas** in the raw text, so they only parse correctly with
a real CSV reader.

In [ ]:
print(raw["history_segment"].value_counts().to_string())
print("\nbucket label vs actual `history` range in that bucket:")
print(raw.groupby("history_segment")["history"].agg(["min", "max", "count"]).to_string())

**Finding.** Seven buckets, no `NaN` — so the embedded commas in
`"6) $750 - $1,000"` and `"7) $1,000 +"` were parsed correctly. Naive
comma-splitting would have corrupted those two rows' worth of customers (3,167 of
them).

The min/max of `history` sit strictly inside each bucket's label range, i.e. the
bucket is a **deterministic function of `history`**. Two consequences:

1. It is redundant as a feature — this is the evidence for excluding
   `history_segment` from `FEATURES` (collinear with `history`).
2. It carries no information `history` lacks, so dropping it loses nothing.

### 4.4 Binary flags — `mens`, `womens`, `newbie`

In [ ]:
for col in ["mens", "womens", "newbie"]:
    vc = raw[col].value_counts()
    print(f"{col:7s} 1={vc.get(1, 0):6d}  0={vc.get(0, 0):6d}  ({vc.get(1, 0) / len(raw):.1%} = 1)")

print("\nmens and womens together:")
print(pd.crosstab(raw["mens"], raw["womens"]).to_string())

**Finding.** Roughly balanced flags: 55% bought mens merchandise, 55% womens,
50% are new. Checking the cross-tab matters because `mens` and `womens` are
**near-complementary** (correlation −0.82): a customer usually bought one
category. They are not redundant, however — a third bought both or neither — so
keeping both is fine.

### 4.5 `zip_code` — and a source typo

In [ ]:
print(raw["zip_code"].value_counts().to_string())
print("\nexpected 3 levels; got:", raw["zip_code"].nunique())

**Finding.** Three levels as expected — but one is misspelled `Surburban` in
**28,776 rows** (the largest level). It is a source-data typo present in every
copy of the file.

**Verdict: normalise on load.** `fetch_hillstrom()` maps it to `Suburban` by
default (`normalize_zip_spelling=True`). Leaving both spellings would split one
category into two and quietly distort any per-zip comparison, including the
fairness check on Day 4.

### 4.6 `channel` — purchase channel

In [ ]:
print(raw["channel"].value_counts().to_string())

**Finding.** Three clean levels: Web (44%), Phone (44%), Multichannel (12%). No
typos, no case problems. `zip_code` and `channel` are the categorical features
that will need encoding (`CATEGORICAL_FEATURES`).

## 5. Outcomes

Three outcomes, in increasing order of rarity: `visit` (did they visit?),
`conversion` (did they buy?), `spend` (how much?). The `spend` distribution drives
almost every modelling decision later.

In [ ]:
summary = pd.DataFrame({
    "n_true": [int(raw[o].sum()) if o != "spend" else int((raw[o] > 0).sum()) for o in OUTCOMES],
    "rate": [raw[o].mean() if o != "spend" else (raw[o] > 0).mean() for o in OUTCOMES],
    "min": [raw[o].min() for o in OUTCOMES],
    "mean": [raw[o].mean() for o in OUTCOMES],
    "max": [raw[o].max() for o in OUTCOMES],
}, index=OUTCOMES)
summary

In [ ]:
print(f"spend == 0 share: {100 * (raw['spend'] == 0).mean():.1f}%")
print("negative spend rows:", int((raw["spend"] < 0).sum()))
print(f"\nspend among the {int((raw['spend'] > 0).sum())} converters:")
print(raw.loc[raw["spend"] > 0, "spend"].describe().to_string())

print("\nspend > 0 exactly when conversion == 1:",
      bool(((raw["spend"] > 0) == (raw["conversion"] == 1)).all()))

ax = raw.loc[raw["spend"] > 0, "spend"].plot.hist(bins=40)
ax.set_xlabel("spend among converters ($)")
ax.set_title("spend is zero for 99.1% of customers; this is the remaining 0.9%")
plt.tight_layout()
plt.show()

**Finding.** Extreme **zero-inflation**: 99.1% of customers spent $0. Among the
578 who converted, spend runs $29.99 → $499.00, mean $116.36.

`spend > 0` holds **exactly** when `conversion == 1` — no exceptions. So `spend`
is `conversion` times a positive amount: it is a post-treatment outcome and a
function of another outcome. It carries no independent information and must never
be a feature.

Two modelling consequences: conversion rates are ~0.9%, so conversion metrics will
be noisy and `visit` (14.7%) is the more stable target to develop against; and any
zero-inflated model of `spend` has to handle the point mass at zero explicitly.

## 6. Relationships between columns

In [ ]:
num = ["recency", "history", "mens", "womens", "newbie",
       "visit", "conversion", "spend"]

raw[num].corr().style.background_gradient(cmap="coolwarm").format("{:.2f}")

In [ ]:
print("numeric correlation, top 6 pairs by |r|:")
corr = raw[num].corr()
pairs = [
    (abs(corr.loc[a, b]), a, b, corr.loc[a, b])
    for i, a in enumerate(num)
    for b in num[i + 1:]
]
for _, a, b, r in sorted(pairs, reverse=True)[:6]:
    print(f"  {a:10s} {b:10s} {r: .2f}")

**Finding.** The strong relationships are exactly the ones to worry about:

- `conversion` ↔ `spend` **0.73** — the leak: same event, two encodings.
- `mens` ↔ `womens` **−0.82** — near-complementary, but not collinear enough to drop.
- `history` ↔ `recency` **−0.25** — recent buyers spent more; real, keep both.
- `visit` ↔ `conversion` **0.23** — today's target is tomorrow's leak; never use
  `visit` to predict `conversion`.

None of the *pre-treatment* features correlate with each other strongly enough to
be redundant, so the feature set can stay as designed.

## 7. Outcome rates by arm (descriptive only)

In [ ]:
from packages.datasets.hillstrom import arm_summary

arm_summary(raw).round(4)

**Finding.** The three arms are close in size (21,306–21,387) and their outcome
rates differ — Mens looks strongest on `visit`. This is *descriptive*: it says the
arms differ, not yet whether the difference is real or noise. Quantifying that is
notebook 02's job (arm choice, SMD, ATE + confidence interval).

## 8. Data-quality verdicts

Everything above collapses into a few decisions, which live in
`packages/datasets/hillstrom.py`:

| Column(s) | Verdict | Why (from this notebook) |
| --- | --- | --- |
| `segment` | **Leaky** — treatment assignment | It *is* the treatment; using it as a feature answers the question before asking it |
| `visit`, `conversion`, `spend` | **Leaky** — post-treatment outcomes | Measured after the e-mail; all three are what we predict |
| `history_segment` | **Excluded** from features | Deterministic function of `history` (§4.3) |
| `zip_code` | Feature, **categorical**; typo fixed | 3 levels, one misspelled (§4.5) |
| `channel` | Feature, **categorical** | 3 clean levels (§4.6) |
| `recency`, `history`, `mens`, `womens`, `newbie` | Features, numeric | No missingness, no sentinel values |
| Duplicate rows | **Keep** | Evenly spread across arms; dropping would change arm sizes (§3) |

`LEAKY_COLUMNS` in the loader is this table's first three rows, and
`FEATURES` is the rest.

## 9. Handoff to notebook 02

Next: `02_hillstrom_randomization_ate.ipynb`, which asks whether the arm
differences in §7 are *real*:

- pick the binary arm (Mens vs No E-Mail) and justify it
- randomization check via standardized mean differences (SMD)
- average treatment effect (ATE) with 95% confidence intervals per outcome

Remaining Day-1 item for you: confirm the profit assumptions in
`configs/business_assumptions.yaml` and the README.